In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

data1 = pd.read_csv('homework_7.1.csv')

In [3]:
data1.head()

,X,W,Z,Y
0,1.137055,1.221768,0.327829,1.944532
1,-0.112905,0.465835,0.599650,0.655514
2,2.077755,1.795414,-0.063393,5.934411
3,0.456373,-0.512159,1.177413,-0.188064
4,-1.012402,0.080002,-0.275697,-0.533775


In [4]:
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
error = np.random.normal(0, 1, (1000,))

Y = X + Z + W + error

correlation = np.corrcoef(X, error)[0, 1]

print(f'The correlation between X and error is {correlation:.3f}')

The correlation between X and error is 0.005


In [6]:
error = W + np.random.normal(0, 1, (1000,))

Y = X + Z + error

correlation = np.corrcoef(X, error)[0, 1]

print(f'The correlation between X and error is {correlation:.3f}')

The correlation between X and error is 0.511


In [8]:
w_values = [-1, 0, 1]

for w in w_values:
    subset = data1[(data1['W'] >= w - 0.1) & (data1['W'] <= w + 0.1)]

    X_reg = subset[['X', 'Z']]
    X_reg = sm.add_constant(X_reg)
    y = subset['Y']

    model = sm.OLS(y, X_reg).fit()

    print(f'Coefficient of X for W {w}: {model.params['X']:.3f}')

Coefficient of X for W -1: 0.858
Coefficient of X for W 0: 1.383
Coefficient of X for W 1: 1.958


In [11]:
def make_error(corr_const, num): 

          err = list() 

          prev = np.random.normal(0, 1) 

          for n in range(num): 

                   prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 

                   err.append(prev) 

          return np.array(err) 

corr_values = [0.2, 0.5, 0.8]

num_obs = 1000
num_trials = 1000

results = {}

for corr in corr_values:
        coefficients = []
        standard_errors = []

        for trial in range(num_trials):

                X = make_error(corr, num_obs)

                error = make_error(corr, num_obs)
                Y = 4 + 2 * X + error

                X = sm.add_constant(X)

                model = sm.OLS(Y, X).fit()

                coefficients.append(model.params[1])
                standard_errors.append(model.bse[1])

        results[corr] = {'SD of X Coefficient': np.std(coefficients, ddof = 1), 'Mean Standard Error' : np.mean(standard_errors)}

results_df = pd.DataFrame(results).T

results_df['ratio'] = results_df['SD of X Coefficient']/results_df['Mean Standard Error']

results_df

,SD of X Coefficient,Mean Standard Error,ratio
0.2,0.032070,0.031673,1.012549
0.5,0.040581,0.031631,1.282979
0.8,0.068760,0.031704,2.168801
